# Program 13: Packet Capture – Network Analysis

## Aim:
Write a Python program using Scapy to capture packets from an authorized interface and display the source address, destination address and protocol.

## Concepts Covered:
* Network interface sniffing and frame processing
* Packet decoding operations (IP Header decoding)
* Hybrid library architecture (Scapy integration with native socket fallbacks)

> **Lab Prerequisite Note:** Packet sniffing utilities require administrative rights. Run your Jupyter Notebook server or IDE terminal **As Administrator** to execute this code successfully.
                                                                                                                                                                        

In [2]:
import socket
import struct
import sys

# Attempt to load Scapy engine cleanly, providing an educational warning if missing
try:
    from scapy.all import sniff, IP
    SCAPY_AVAILABLE = True
except ImportError:
    SCAPY_AVAILABLE = False

def run_scapy_sniffer(packet_count=5):
    """Captures and decodes packet fields using the Scapy toolkit layer."""
    print(f"[Engine] Starting Scapy Sniffer Core... Capturing {packet_count} packets...")
    
    def process_packet(packet):
        # Verify if the frame contains a valid network layer payload layout
        if packet.haslayer(IP):
            src_ip = packet[IP].src
            dst_ip = packet[IP].dst
            proto = packet[IP].proto
            
            # Map standard protocol numbers to clear, readable strings
            proto_name = "TCP" if proto == 6 else "UDP" if proto == 17 else "ICMP" if proto == 1 else f"RAW-{proto}"
            print(f" -> [Scapy Cap] Src: {src_ip:<15} | Dst: {dst_ip:<15} | Proto: {proto_name}")

    # Fire off the packet extraction engine loop
    sniff(prn=process_packet, count=packet_count, timeout=10)

def run_native_fallback_sniffer(packet_count=5):
    """Alternative raw socket decoder engine if Scapy environment configurations are missing."""
    print(f"[Engine] Scapy fallback triggered. Launching Native RAW Socket Sniffer...")
    
    try:
        # Determine current host OS environment settings to allocate the correct socket type
        if sys.platform == "win32":
            # Windows raw sockets require binding to an active interface
            hostname = socket.gethostname()
            host_ip = socket.gethostbyname(hostname)
            sock_type = socket.AF_INET
            
            sniffer = socket.socket(sock_type, socket.SOCK_RAW, socket.IPPROTO_IP)
            sniffer.bind((host_ip, 0))
            # Enable promiscuous mode setup flags on Windows interfaces
            sniffer.ioctl(socket.SIO_RCVALL, socket.RCVALL_ON)
        else:
            # Linux/Unix standard raw interface bindings
            sniffer = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_TCP)

        sniffer.settimeout(5.0)
        print(f" -> Sniffing active network packets on local boundary line...\n")
        
        for _ in range(packet_count):
            try:
                raw_packet, addr = sniffer.recvfrom(65565)
                
                # Unpack the standard IPv4 header fields (First 20 bytes contain addressing data)
                ip_header = raw_packet[0:20]
                iph = struct.unpack('!BBHHHBBH4s4s', ip_header)
                
                protocol_code = iph[6]
                src_ip = socket.inet_ntoa(iph[8])
                dst_ip = socket.inet_ntoa(iph[9])
                
                proto_name = "TCP" if protocol_code == 6 else "UDP" if protocol_code == 17 else f"RAW-{protocol_code}"
                print(f" -> [Native Cap] Src: {src_ip:<15} | Dst: {dst_ip:<15} | Proto: {proto_name}")
            except socket.timeout:
                print(" -> [Timeout] No packet traffic captured within the listening window.")
                break

        # Disable promiscuous mode flags upon graceful termination
        if sys.platform == "win32":
            sniffer.ioctl(socket.SIO_RCVALL, socket.RCVALL_OFF)
            
    except PermissionError:
        print("\n[Access Error] Administrative rights required! Close Jupyter and relaunch it 'As Administrator'.")
    except Exception as e:
        print(f"\n[System Error] Could not initialize raw socket channels: {e}")

if __name__ == "__main__":
    print("=== PACKET CAPTURE & ANALYSIS LAB ===")
    CAPTURE_LIMIT = 5
    
    if SCAPY_AVAILABLE:
        try:
            run_scapy_sniffer(packet_count=CAPTURE_LIMIT)
        except Exception as scapy_err:
            print(f"[Warning] Scapy hit a local driver error: {scapy_err}")
            run_native_fallback_sniffer(packet_count=CAPTURE_LIMIT)
    else:
        print("[System Info] Scapy package is not installed in this environment.")
        run_native_fallback_sniffer(packet_count=CAPTURE_LIMIT)
        
    print("\nCapture session complete.")


=== PACKET CAPTURE & ANALYSIS LAB ===
[System Info] Scapy package is not installed in this environment.
[Engine] Scapy fallback triggered. Launching Native RAW Socket Sniffer...

[Access Error] Administrative rights required! Close Jupyter and relaunch it 'As Administrator'.

Capture session complete.
